In [1]:
import os
import numpy as np
import pandas as pd
from config import CLEAN_DATA_PATH ,RANDOM_STATE ,BINARY_COLS ,MULTI_CAT_COLS ,NUMERIC_COLS ,REGRESSION_TARGET ,CLASSIFICATION_TARGET ,DASHBOARD_RESOURCES_PATH

# Preproccsing 
from sklearn.model_selection import train_test_split ,GridSearchCV ,KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler , OneHotEncoder , PolynomialFeatures

# Models
from sklearn.linear_model import LinearRegression ,Ridge ,Lasso ,LogisticRegression 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC 
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier , GradientBoostingClassifier , RandomForestRegressor ,GradientBoostingRegressor

# Metics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
df = pd.read_csv(CLEAN_DATA_PATH)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Gender             7043 non-null   str    
 1   Senior Citizen     7043 non-null   str    
 2   Partner            7043 non-null   str    
 3   Dependents         7043 non-null   str    
 4   Tenure Months      7043 non-null   int64  
 5   Phone Service      7043 non-null   str    
 6   Multiple Lines     7043 non-null   str    
 7   Internet Service   7043 non-null   str    
 8   Online Security    7043 non-null   str    
 9   Online Backup      7043 non-null   str    
 10  Device Protection  7043 non-null   str    
 11  Tech Support       7043 non-null   str    
 12  Streaming TV       7043 non-null   str    
 13  Streaming Movies   7043 non-null   str    
 14  Contract           7043 non-null   str    
 15  Paperless Billing  7043 non-null   str    
 16  Payment Method     7043 non-null   

In [4]:
cls_drop = ["Churn Value"]
reg_drop = ["CLTV", "Churn Value"]

In [5]:
df.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value,CLTV
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,3239
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,2701
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1,5372
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1,5003
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1,5340


In [6]:
def make_preprocessor():
    return ColumnTransformer(transformers=[
        ("num", StandardScaler(), NUMERIC_COLS),
        ("bin", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), BINARY_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), MULTI_CAT_COLS),
    ])


In [7]:
def make_poly_preprocessor():
    numeric_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(include_bias=False)),
    ])
    return ColumnTransformer(transformers=[
        ("num", numeric_pipe, NUMERIC_COLS),
        ("bin", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), BINARY_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), MULTI_CAT_COLS),
    ])

In [8]:
regression_results = []
classification_results = []


In [9]:
def log_regression_result(results_list, model_name, y_true, y_pred, best_params=None):
    row = {
        "Model": model_name,
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Best_Params": best_params if best_params is not None else "-",
    }
    results_list.append(row)
    print(f"{model_name:30s} | RMSE: {row['RMSE']:8.2f} | MAE: {row['MAE']:8.2f} | "
          f"R2: {row['R2']:.4f} | Params: {row['Best_Params']}")
    return row


# Reggression 
### Target value CLTV

Models 
1. linear
2. Lasso
3. Ridge
4. Random forest Reggresion 
5. Gradiant boosting Reggresion

In [10]:
reg_feature_cols = NUMERIC_COLS + BINARY_COLS + MULTI_CAT_COLS
X_reg = df[reg_feature_cols]
y_reg = df[REGRESSION_TARGET]
 
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

In [11]:
def train_linear_regression(X_train, X_test, y_train, y_test, results_list):
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", LinearRegression()),
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
 
    log_regression_result(results_list, "Linear Regression", y_test, preds)
    return pipeline


In [12]:
def train_polynomial_regression(X_train, X_test, y_train, y_test, results_list):
    pipeline = Pipeline([
        ("preprocessor", make_poly_preprocessor()),
        ("model", LinearRegression()),
    ])
    param_grid = {"preprocessor__num__poly__degree": [1, 2, 3]}
 
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="r2",
        cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
 
    best_pipeline = search.best_estimator_
    preds = best_pipeline.predict(X_test)
 
    log_regression_result(results_list, "Polynomial Regression", y_test, preds,
                           best_params=search.best_params_)
    return best_pipeline


In [13]:
def train_lasso_regression(X_train, X_test, y_train, y_test, results_list):
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", Lasso(random_state=RANDOM_STATE, max_iter=10000)),
    ])
    param_grid = {"model__alpha": [0.001, 0.01, 0.1, 1.0, 10.0, 50.0]}
 
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="r2",
        cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
 
    best_pipeline = search.best_estimator_
    preds = best_pipeline.predict(X_test)
 
    log_regression_result(results_list, "Lasso Regression", y_test, preds,
                           best_params=search.best_params_)
    return best_pipeline


In [14]:
def train_ridge_regression(X_train, X_test, y_train, y_test, results_list):
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", Ridge(random_state=RANDOM_STATE)),
    ])
    param_grid = {"model__alpha": [0.01, 0.1, 1.0, 10.0, 50.0, 100.0]}
 
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="r2",
        cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
 
    best_pipeline = search.best_estimator_
    preds = best_pipeline.predict(X_test)
 
    log_regression_result(results_list, "Ridge Regression", y_test, preds,
                           best_params=search.best_params_)
    return best_pipeline


In [15]:
def train_random_forest_regressor(X_train, X_test, y_train, y_test, results_list):
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
    ])
    param_grid = {
        "model__n_estimators": [100, 300],
        "model__max_depth": [5, 10, None],
        "model__min_samples_leaf": [1, 5, 10],
    }
 
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="r2",
        cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
 
    best_pipeline = search.best_estimator_
    preds = best_pipeline.predict(X_test)
 
    log_regression_result(results_list, "Random Forest Regressor", y_test, preds,
                           best_params=search.best_params_)
    return best_pipeline


In [16]:
def train_gradient_boosting_regressor(X_train, X_test, y_train, y_test, results_list):
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", GradientBoostingRegressor(random_state=RANDOM_STATE)),
    ])
    param_grid = {
        "model__n_estimators": [100, 300],
        "model__learning_rate": [0.01, 0.1],
        "model__max_depth": [2, 3, 4],
    }
 
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="r2",
        cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
 
    best_pipeline = search.best_estimator_
    preds = best_pipeline.predict(X_test)
 
    log_regression_result(results_list, "Gradient Boosting Regressor", y_test, preds,
                           best_params=search.best_params_)
    return best_pipeline


In [17]:
train_linear_regression(X_reg_train, X_reg_test, y_reg_train, y_reg_test, regression_results)
train_polynomial_regression(X_reg_train, X_reg_test, y_reg_train, y_reg_test, regression_results)
train_lasso_regression(X_reg_train, X_reg_test, y_reg_train, y_reg_test, regression_results)
train_ridge_regression(X_reg_train, X_reg_test, y_reg_train, y_reg_test, regression_results)
train_random_forest_regressor(X_reg_train, X_reg_test, y_reg_train, y_reg_test, regression_results)
train_gradient_boosting_regressor(X_reg_train, X_reg_test, y_reg_train, y_reg_test, regression_results)

pd.DataFrame(regression_results).to_csv("regression_results.csv", index=False)


Linear Regression              | RMSE:  1062.44 | MAE:   897.94 | R2: 0.1691 | Params: -
Polynomial Regression          | RMSE:  1046.15 | MAE:   887.26 | R2: 0.1944 | Params: {'preprocessor__num__poly__degree': 3}
Lasso Regression               | RMSE:  1061.68 | MAE:   897.94 | R2: 0.1703 | Params: {'model__alpha': 10.0}
Ridge Regression               | RMSE:  1061.99 | MAE:   898.22 | R2: 0.1698 | Params: {'model__alpha': 100.0}
Random Forest Regressor        | RMSE:  1026.39 | MAE:   873.40 | R2: 0.2245 | Params: {'model__max_depth': 5, 'model__min_samples_leaf': 5, 'model__n_estimators': 300}
Gradient Boosting Regressor    | RMSE:  1026.25 | MAE:   873.28 | R2: 0.2247 | Params: {'model__learning_rate': 0.01, 'model__max_depth': 2, 'model__n_estimators': 300}


In [9]:
regression_models = {
    "Linear Regression": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", LinearRegression()),
    ]),
    "Lasso Regression": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", Lasso(random_state=RANDOM_STATE, max_iter=10000)),
    ]),
    "Ridge Regression": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", Ridge(random_state=RANDOM_STATE)),
    ]),
    "Random Forest Regressor": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    "Gradient Boosting Regressor": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", GradientBoostingRegressor(random_state=RANDOM_STATE)),
    ]),
}


In [10]:
for name, pipe in regression_models.items():
    pipe.fit(X_reg_train, y_reg_train)
    preds = pipe.predict(X_reg_test)
 
    rmse = mean_squared_error(y_reg_test, preds) ** 0.5
    mae = mean_absolute_error(y_reg_test, preds)
    r2 = r2_score(y_reg_test, preds)
 
    print(f"{name:30s} | RMSE: {rmse:8.2f} | MAE: {mae:8.2f} | R2: {r2:.4f}")
 
    regression_results.append({"Model": name, "RMSE": rmse, "MAE": mae, "R2": r2})

Linear Regression              | RMSE:  1062.44 | MAE:   897.94 | R2: 0.1691
Lasso Regression               | RMSE:  1062.03 | MAE:   898.16 | R2: 0.1697
Ridge Regression               | RMSE:  1062.32 | MAE:   897.96 | R2: 0.1693
Random Forest Regressor        | RMSE:  1074.72 | MAE:   905.60 | R2: 0.1498
Gradient Boosting Regressor    | RMSE:  1032.83 | MAE:   876.11 | R2: 0.2147


# Classification 
### Target value Churn

Models 
1. Logistic
2. KNN
3. SVM
4. Descion tree
5. Random forest
6. Gradiant boosting 

In [11]:
clf_feature_cols = NUMERIC_COLS + BINARY_COLS + MULTI_CAT_COLS
X_clf = df[clf_feature_cols]
y_clf = df[CLASSIFICATION_TARGET]
 
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=RANDOM_STATE, stratify=y_clf
)


In [17]:
classification_models = {
    "Logistic Regression":Pipeline([
        ("preprocessor",make_preprocessor()),
        ("model",LogisticRegression())
    ]),
    "KNN": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", KNeighborsClassifier()),
    ]),
    "SVM": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", SVC(probability=True, random_state=RANDOM_STATE)),
    ]),
    "Decision Tree": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    "Gradient Boosting": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ]),
}


In [18]:
for name, pipe in classification_models.items():
    pipe.fit(X_clf_train, y_clf_train)
    preds = pipe.predict(X_clf_test)
    probs = pipe.predict_proba(X_clf_test)[:, 1]
 
    acc = accuracy_score(y_clf_test, preds)
    prec = precision_score(y_clf_test, preds)
    rec = recall_score(y_clf_test, preds)
    f1 = f1_score(y_clf_test, preds)
    auc = roc_auc_score(y_clf_test, probs)
 
    print(f"{name:20s} | Acc: {acc:.2f} | Prec: {prec:.2f} | Rec: {rec:.2f} | "
          f"F1: {f1:.2f} | AUC: {auc:.2f}")
 
    classification_results.append({
        "Model": name, "Accuracy": acc, "Precision": prec,
        "Recall": rec, "F1": f1, "ROC_AUC": auc,
    })


Logistic Regression  | Acc: 0.80 | Prec: 0.64 | Rec: 0.57 | F1: 0.60 | AUC: 0.85
KNN                  | Acc: 0.77 | Prec: 0.57 | Rec: 0.57 | F1: 0.57 | AUC: 0.79
SVM                  | Acc: 0.80 | Prec: 0.65 | Rec: 0.52 | F1: 0.58 | AUC: 0.81
Decision Tree        | Acc: 0.75 | Prec: 0.52 | Rec: 0.56 | F1: 0.54 | AUC: 0.69
Random Forest        | Acc: 0.79 | Prec: 0.63 | Rec: 0.50 | F1: 0.56 | AUC: 0.83
Gradient Boosting    | Acc: 0.80 | Prec: 0.65 | Rec: 0.53 | F1: 0.59 | AUC: 0.85


In [14]:
pd.DataFrame(regression_results).to_csv(
    os.path.join(DASHBOARD_RESOURCES_PATH, "regression_results.csv"), index=False
)
pd.DataFrame(classification_results).to_csv(
    os.path.join(DASHBOARD_RESOURCES_PATH, "classification_results.csv"), index=False
)
